# 1. Setup and Libraries

In [1]:
import os
import glob
import json
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import time

# 2. Dataset Directories

In [2]:
dataset_Folder = "G:/My Drive/GP Datasets/fit3d_data/"
file_path = dataset_Folder + 'fit3d_info.json'

# 3. Load FIT3D Metadata

In [3]:
info_JSON = pd.read_json(file_path, typ='series')
print("JSON loaded successfully as a series:")
print(info_JSON.head())

val_subj_names = ['s03', 's11']
all_train = [sub for sub in info_JSON['train_subj_names'] if sub not in val_subj_names]
all_camera_names = info_JSON['all_camera_names']

# Extract all actions from all subjects and find unique values
all_actions_list = [action for actions in info_JSON['subj_to_act'].values() for action in actions]
unique_actions = sorted(list(set(all_actions_list)))

print(f"Number of unique actions: {len(unique_actions)}")
print("Unique actions:")
print(unique_actions)

JSON loaded successfully as a series:
subj_to_act         {'s03': ['band_pull_apart', 'dumbbell_high_pul...
test_subj_names                                       [s02, s12, s13]
train_subj_names             [s03, s04, s05, s07, s08, s09, s10, s11]
all_camera_names             [50591643, 58860488, 60457274, 65906101]
dtype: object
Number of unique actions: 47
Unique actions:
['band_pull_apart', 'barbell_dead_row', 'barbell_row', 'barbell_shrug', 'burpees', 'clean_and_press', 'deadlift', 'diamond_pushup', 'drag_curl', 'dumbbell_biceps_curls', 'dumbbell_curl_trifecta', 'dumbbell_hammer_curls', 'dumbbell_high_pulls', 'dumbbell_overhead_shoulder_press', 'dumbbell_reverse_lunge', 'dumbbell_scaptions', 'man_maker', 'mule_kick', 'neutral_overhead_shoulder_press', 'one_arm_row', 'overhead_extension_thruster', 'overhead_trap_raises', 'pushup', 'side_lateral_raise', 'squat', 'standing_ab_twists', 'w_raise', 'walk_the_box', 'warmup_1', 'warmup_10', 'warmup_11', 'warmup_12', 'warmup_13', 'warmup_14

# 10. Repetition Segmentation & Counting Evaluation
Evaluates the repetition-detection pipeline using **joints3d_25** 3D skeletons 
from the Fit3D dataset as input.  
Ground truth is sourced from `rep_ann.json` which stores a list of frame-boundary 
indices per action.

**Metrics:**
- **Repetition Segmentation:** Intersection-over-Union (IoU) between predicted and 
annotated repetition intervals (averaged over repetitions)
- **Repetition Counting:** Off-By-One error rate (OBO) and Mean Absolute Error (MAE) of counts

## 10.1 FIT3D Joint Mapping & Exercise Name Mapping

In [4]:
import sys
sys.path.append('d:/GP/Pose/Code')
from exercise_config import EXERCISE_TO_CONFIG, JOINT_DEFINITIONS

# Fit3D joints3d_25 index mapping (left_idx, right_idx)
# FIT3D skeleton: 0=neck, 1=L_hip, 2=L_knee, 3=L_ankle, 4=R_hip,
#   5=R_knee, 6=R_ankle, 7=spine, 8=spine2, 9=head,
#   10=thorax, 11=L_shoulder, 12=L_elbow, 13=L_wrist,
#   14=R_shoulder, 15=R_elbow, 16=R_wrist, 17=L_thumb,
#   18=L_index, 19=L_pinky, 20=R_thumb, 21=R_index, 22=R_pinky,
#   23=L_heel, 24=R_heel
FIT3D_JOINT_MAP = {
    'hip':        (1, 4),
    'knee':       (2, 5),
    'ankle':      (3, 6),
    'shoulder':   (11, 14),
    'elbow':      (12, 15),
    'wrist':      (13, 16),
    'index':      (18, 21),
    'nose':       None,       # no direct equivalent — skipped
    'vertical':   None,       # virtual reference — handled specially
    'horizontal': None,       # virtual reference — handled specially
}

# Map Fit3D action names -> exercise_config keys
FIT3D_TO_CONFIG_NAME = {
    'squat':                           'squat',
    'deadlift':                        'deadlift',
    'pushup':                          'push-up',
    'diamond_pushup':                  'push-up',
    'dumbbell_biceps_curls':           'barbell biceps curl',
    'dumbbell_hammer_curls':           'hammer curl',
    'drag_curl':                       'barbell biceps curl',
    'barbell_dead_row':                'deadlift',
    'dumbbell_reverse_lunge':          'squat',
    'dumbbell_overhead_shoulder_press':'shoulder press',
    'neutral_overhead_shoulder_press': 'shoulder press',
    'dumbbell_high_pulls':             'lat pulldown',
    'overhead_extension_thruster':     'shoulder press',
    'side_lateral_raise':              'lateral raise',
    'dumbbell_scaptions':              'lateral raise',
    'one_arm_row':                     'lat pulldown',
    'barbell_row':                     'lat pulldown',
    'overhead_trap_raises':            'lateral raise',
    'w_raise':                         'lateral raise',
    'barbell_shrug':                   'lat pulldown',
    'band_pull_apart':                 'lat pulldown',
    'mule_kick':                       'squat',
    'burpees':                         'push-up',
    'clean_and_press':                 'shoulder press',
    'dumbbell_curl_trifecta':          'barbell biceps curl',
    'man_maker':                       'push-up',
    'standing_ab_twists':              'russian twist',
    'walk_the_box':                    'squat',
}  # warmup_* actions excluded — no reliable rep metric mapping

print(f"Mapped {len(FIT3D_TO_CONFIG_NAME)} Fit3D actions to exercise configs.")

Mapped 28 Fit3D actions to exercise configs.


## 10.2 Angle Extraction from joints3d_25

In [5]:
def calculate_angle_3d(a, b, c):
    """Computes the angle at joint B (in degrees) between vectors BA and BC."""
    ba = a - b
    bc = c - b
    n1 = np.linalg.norm(ba)
    n2 = np.linalg.norm(bc)
    if n1 < 1e-6 or n2 < 1e-6:
        return np.nan
    cosine_angle = np.dot(ba, bc) / (n1 * n2)
    return np.degrees(np.arccos(np.clip(cosine_angle, -1.0, 1.0)))


def extract_angle_sequence(joints, metric_name, metric_config):
    """
    Given a joints3d_25 array of shape [T, 25, 3], extract the angle
    sequence for the specified metric, averaged over left and right sides.
    Returns np.array of shape [T] (may contain NaN for frames with no data).
    """
    j_names = metric_config['joints']  # e.g. ['shoulder', 'elbow', 'wrist']

    # Quickly reject metrics that require nose (unavailable in FIT3D)
    if 'nose' in j_names:
        return None

    T = len(joints)
    angles = np.full(T, np.nan)

    for fi in range(T):
        frame = joints[fi]
        side_angles = []

        for side_idx in [0, 1]:  # 0 = left, 1 = right
            pts = []
            valid = True
            for jn in j_names:
                if jn in ('vertical', 'horizontal'):
                    pts.append(jn)  # placeholder — resolved below
                elif FIT3D_JOINT_MAP.get(jn) is None:
                    valid = False
                    break
                else:
                    idx = FIT3D_JOINT_MAP[jn][side_idx]
                    pts.append(frame[idx].copy())

            if not valid or len(pts) != 3:
                continue

            # Find a real anchor point to resolve virtual references
            mid = next((p for p in pts if isinstance(p, np.ndarray)), None)
            if mid is None:
                continue

            resolved = []
            for p in pts:
                if isinstance(p, str):
                    if p == 'vertical':
                        resolved.append(np.array([mid[0], mid[1] - 0.5, mid[2]]))
                    else:  # 'horizontal'
                        resolved.append(np.array([mid[0] + 0.5, mid[1], mid[2]]))
                else:
                    resolved.append(p)

            ang = calculate_angle_3d(resolved[0], resolved[1], resolved[2])
            if not np.isnan(ang):
                side_angles.append(ang)

        if side_angles:
            angles[fi] = np.mean(side_angles)

    return angles


def get_primary_metric(action_name):
    """Returns (metric_name, metric_config, rep_low, rep_high) for an action, or None."""
    config_name = FIT3D_TO_CONFIG_NAME.get(action_name)
    if config_name is None or config_name not in EXERCISE_TO_CONFIG:
        return None

    ex_config = EXERCISE_TO_CONFIG[config_name]
    ref       = ex_config.get('thresholds', {})

    for m in ex_config.get('metrics', []):
        if m not in JOINT_DEFINITIONS:
            continue
        mdef = JOINT_DEFINITIONS[m]
        if mdef.get('type', 'angle') != 'angle':
            continue
        if 'nose' in mdef.get('joints', []):
            continue
        rmin = ref.get(f"{m}_min")
        rmax = ref.get(f"{m}_max")
        if rmin is None or rmax is None or (rmax - rmin) <= 0:
            continue
        span = rmax - rmin
        return m, mdef, rmin + 0.4 * span, rmax - 0.4 * span

    return None


print("Angle extraction functions defined.")

Angle extraction functions defined.


## 10.3 Repetition Boundary Detection

In [6]:
from scipy.signal import savgol_filter


def detect_rep_boundaries(angles, rep_low, rep_high, min_rep_frames=5):
    """
    Detects repetition segment boundaries from an angle sequence.

    Uses the same extended → contracted → extended state-machine as inference.py.
    Each completed cycle = one repetition.

    Returns:
        boundaries: list of frame indices (first element = 0, last = T-1).
                    The number of repetitions = len(boundaries) - 2.
    """
    valid_mask = ~np.isnan(angles)
    if valid_mask.sum() < 10:
        return [0, len(angles) - 1]

    # Infer value at NaN frames by linear interpolation
    interp_angles = angles.copy()
    if not valid_mask.all():
        interp_angles = np.interp(
            np.arange(len(angles)),
            np.where(valid_mask)[0],
            angles[valid_mask]
        )

    # Savitzky-Golay smoothing — reduces MediaPipe/sensor jitter
    win = min(11, int(valid_mask.sum() / 4) * 2 + 1)
    win = max(win, 3)
    try:
        smoothed = savgol_filter(interp_angles, win, 2)
    except Exception:
        smoothed = interp_angles

    # State-machine
    phase = 'extended'
    boundaries = [0]
    last_boundary = 0

    for i, val in enumerate(smoothed):
        if phase == 'extended' and val < rep_low:
            phase = 'contracted'
        elif phase == 'contracted' and val > rep_high:
            phase = 'extended'
            if i - last_boundary >= min_rep_frames:
                boundaries.append(i)
                last_boundary = i

    boundaries.append(len(angles) - 1)
    return boundaries


print("Repetition boundary detector defined.")

Repetition boundary detector defined.


## 10.4 Metric Functions (IoU, OBO, MAE)

In [7]:
def intervals_from_bounds(bounds):
    """Convert boundary list [b0, b1, ..., bn] into (len-1) intervals."""
    return [(bounds[i], bounds[i + 1]) for i in range(len(bounds) - 1)]


def iou_interval(a, b):
    """IoU between two 1-D intervals a=(s1,e1) and b=(s2,e2)."""
    inter_start = max(a[0], b[0])
    inter_end   = min(a[1], b[1])
    if inter_end <= inter_start:
        return 0.0
    intersection = inter_end - inter_start
    union = (a[1] - a[0]) + (b[1] - b[0]) - intersection
    return intersection / union if union > 0 else 0.0


def mean_iou_sequence(gt_bounds, pred_bounds):
    """
    Compute mean IoU between GT and predicted repetition intervals.

    Each GT interval is matched to the best-IoU predicted interval (greedy).
    Unmatched GT intervals contribute IoU = 0.
    """
    gt_ivs   = intervals_from_bounds(gt_bounds)
    pred_ivs = intervals_from_bounds(pred_bounds)

    if not gt_ivs:
        return np.nan
    if not pred_ivs:
        return 0.0

    ious     = []
    used_pred = set()
    for gt_iv in gt_ivs:
        best_iou, best_j = 0.0, -1
        for j, pred_iv in enumerate(pred_ivs):
            if j in used_pred:
                continue
            s = iou_interval(gt_iv, pred_iv)
            if s > best_iou:
                best_iou, best_j = s, j
        if best_j >= 0:
            used_pred.add(best_j)
        ious.append(best_iou)

    return float(np.mean(ious))


print("Metric functions (IoU, OBO, MAE) defined.")

Metric functions (IoU, OBO, MAE) defined.


## 10.5 Full Evaluation Loop

In [8]:
def evaluate_rep_counting(dataset_base, subj_names, min_rep_frames=5):
    """
    Full repetition segmentation & counting evaluation.

    Args:
        dataset_base   : root path of fit3d_data
        subj_names     : list of subject IDs to evaluate
        min_rep_frames : minimum frames per repetition (prevents spurious counts)

    Returns:
        results_df : pd.DataFrame with one row per (subject, action)
        summary    : dict with overall OBO, MAE, mean IoU
    """
    rows = []
    base = Path(dataset_base)

    for subj in subj_names:
        rep_ann_path = base / 'train' / subj / 'rep_ann.json'
        if not rep_ann_path.exists():
            print(f"[WARN] No rep_ann.json for {subj}")
            continue

        with open(rep_ann_path) as f:
            rep_ann = json.load(f)

        joints_dir = base / 'train' / subj / 'joints3d_25'
        for joint_file in sorted(joints_dir.glob('*.json')):
            action = joint_file.stem

            # Need GT bounds and a metric mapping
            if action not in rep_ann:
                continue
            gt_bounds = rep_ann[action]
            if len(gt_bounds) < 2:
                continue
            gt_count = len(gt_bounds) - 1

            pm = get_primary_metric(action)
            if pm is None:
                continue
            metric_name, metric_config, rep_low, rep_high = pm

            # Load 3-D joints
            with open(joint_file) as f:
                data = json.load(f)
            joints = (np.array(data.get('joints3d_25', list(data.values())[0]))
                      if isinstance(data, dict) else np.array(data))

            # Compute angle curve
            angles = extract_angle_sequence(joints, metric_name, metric_config)
            if angles is None or np.isnan(angles).all():
                continue

            # Detect predicted boundaries
            pred_bounds = detect_rep_boundaries(
                angles, rep_low, rep_high, min_rep_frames=min_rep_frames)
            # Boundaries: [0, b1, b2, ..., T-1]
            # Rep count = number of interior boundaries = len - 2
            pred_count = max(0, len(pred_bounds) - 2)

            # IoU: use interior + last boundaries as predicted rep intervals
            # GT bounds come from rep_ann as-is (already frame indices)
            pred_iv_bounds = [pred_bounds[0]] + pred_bounds[1:-1] + [pred_bounds[-1]]
            miou = mean_iou_sequence(gt_bounds, pred_iv_bounds)

            abs_err = abs(gt_count - pred_count)
            obo     = int(abs_err <= 1)

            rows.append({
                'subject':     subj,
                'action':      action,
                'metric_used': metric_name,
                'gt_count':    gt_count,
                'pred_count':  pred_count,
                'abs_error':   abs_err,
                'obo':         obo,
                'mean_iou':    miou,
            })

    if not rows:
        print("No evaluable sequences found.")
        return pd.DataFrame(), {}

    results_df = pd.DataFrame(rows)
    summary = {
        'Total Sequences': len(results_df),
        'MAE':             results_df['abs_error'].mean(),
        'OBO Rate (%)':    results_df['obo'].mean() * 100,
        'Mean IoU':        results_df['mean_iou'].dropna().mean(),
    }
    return results_df, summary


print("Evaluation loop defined.")

Evaluation loop defined.


## 10.6 Run Evaluation (Validation Subjects)

In [9]:
rep_results_df, rep_summary = evaluate_rep_counting(
    dataset_Folder,
    val_subj_names
)

if not rep_results_df.empty:
    print("\n=== Per-Action Results ===")
    display(rep_results_df.sort_values(['subject', 'action']).reset_index(drop=True))

    print("\n=== Per-Exercise Summary ===")
    per_action = rep_results_df.groupby('action').agg(
        GT_Count_Mean=('gt_count',   'mean'),
        Pred_Count_Mean=('pred_count','mean'),
        MAE=('abs_error',            'mean'),
        OBO_Rate=('obo',             lambda x: x.mean() * 100),
        Mean_IoU=('mean_iou',        'mean'),
    ).round(3)
    display(per_action)

    print("\n=== Overall Summary ===")
    for k, v in rep_summary.items():
        print(f"  {k:25s}: {v:.4f}" if isinstance(v, float) else f"  {k:25s}: {v}")


=== Per-Action Results ===


,subject,action,metric_used,gt_count,pred_count,abs_error,obo,mean_iou
0,s03,band_pull_apart,ELBOW_ANGLE,5,0,5,0,0.033475
1,s03,barbell_dead_row,HIP_EXTENSION,5,4,1,1,0.210835
2,s03,barbell_row,ELBOW_ANGLE,5,5,0,1,0.550232
3,s03,barbell_shrug,ELBOW_ANGLE,5,0,5,0,0.022897
4,s03,burpees,ELBOW_ANGLE,5,5,0,1,0.365523
5,s03,clean_and_press,ELBOW_ANGLE,6,12,6,0,0.413975
6,s03,deadlift,HIP_EXTENSION,5,7,2,0,0.343935
7,s03,diamond_pushup,ELBOW_ANGLE,5,5,0,1,0.670352
8,s03,drag_curl,ELBOW_ANGLE,5,5,0,1,0.461524
9,s03,dumbbell_biceps_curls,ELBOW_ANGLE,5,5,0,1,0.530426



=== Per-Exercise Summary ===


,GT_Count_Mean,Pred_Count_Mean,MAE,OBO_Rate,Mean_IoU
action,,,,,
band_pull_apart,5.0,0.0,5.0,0.0,0.032
barbell_dead_row,5.0,5.0,1.0,100.0,0.398
barbell_row,5.0,5.0,0.0,100.0,0.555
barbell_shrug,5.0,0.0,5.0,0.0,0.021
burpees,5.5,5.5,0.0,100.0,0.384
clean_and_press,5.5,11.0,5.5,0.0,0.477
deadlift,5.0,6.5,1.5,50.0,0.475
diamond_pushup,5.5,5.5,0.0,100.0,0.684
drag_curl,5.0,5.0,0.0,100.0,0.550



=== Overall Summary ===
  Total Sequences          : 54
  MAE                      : 2.5000
  OBO Rate (%)             : 64.8148
  Mean IoU                 : 0.4097
